## Google Colab Setup

This notebook can run either locally from the repo or on Google Colab.

Colab workflow:
- Mount Google Drive and save all trained-model outputs there immediately.
- Download only the exact code and data files needed into `/content/XAllergen2.0`.
- Download a fresh ESM-2 snapshot from Hugging Face.

Required GitHub-hosted artifacts:
- `models/baseline_frozen_esm2.pt`
- `data/positives_splitA.csv`
- `data/positives_splitB.csv`
- `data/negatives_splitA.csv`
- `data/negatives_splitB.csv`

Section C reads `results/rsa_regularization/sweep_summary.csv` (notebook 12) and
`results/classification/mtl_baseline_metrics.json` (notebook 04) for comparison.

**Regularization formula**

$$\mathcal{L} = \lambda_\text{cls}\mathcal{L}_\text{cls}
  + \lambda_\text{reg}\,\mathbf{1}[y=1]\,\frac{1}{L}\sum_{i=1}^{L}\alpha_i(1-e_i)$$

where $e_i \in \{0, 1\}$ is the binary per-residue IEDB epitope label
(1 = residue covered by at least one experimentally validated epitope interval,
0 = non-epitope). The $\mathbf{1}[y=1]$ masking is enforced implicitly:
non-allergen proteins receive no IEDB feature vector and are excluded from the
regularization loss automatically.

This is the attention-regularization counterpart to notebook 04 (MTL epitope supervision).
Both use the same `positives_splitA` training data and `baseline_frozen_esm2.pt` starting
checkpoint; the difference is that nb 04 adds a per-residue epitope prediction head while
this notebook constrains the existing pooling attention directly.

In [ ]:
import os
import sys
import urllib.request
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules or Path('/content').exists()
os.environ['XALLERGEN_RUN_TARGET'] = 'colab' if IS_COLAB else 'local'

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/XAllergen2.0')
else:
    DRIVE_ROOT = None

if IS_COLAB:
    RUNTIME_ROOT = Path('/content/XAllergen2.0')
else:
    for _candidate in [Path.cwd(), *Path.cwd().parents]:
        if (_candidate / 'src' / 'xallergen').exists():
            RUNTIME_ROOT = _candidate
            break
    else:
        raise FileNotFoundError('Could not locate repo root with src/xallergen')

SRC_DIR             = RUNTIME_ROOT / 'src'
PACKAGE_DIR         = SRC_DIR / 'xallergen'
DATA_DIR            = RUNTIME_ROOT / 'data'
RUNTIME_MODELS_DIR  = RUNTIME_ROOT / 'models'
RUNTIME_RESULTS_DIR = RUNTIME_ROOT / 'results'
HF_DIR              = RUNTIME_ROOT / 'hf_models' / 'facebook_esm2_t6_8M_UR50D'

if IS_COLAB:
    MODELS_DIR  = DRIVE_ROOT / 'models'
    RESULTS_DIR = DRIVE_ROOT / 'results'
else:
    MODELS_DIR  = RUNTIME_MODELS_DIR
    RESULTS_DIR = RUNTIME_RESULTS_DIR

for path in [
    SRC_DIR, PACKAGE_DIR, DATA_DIR,
    RUNTIME_MODELS_DIR, RUNTIME_RESULTS_DIR, MODELS_DIR, RESULTS_DIR, HF_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

if IS_COLAB:
    from huggingface_hub import snapshot_download

    RAW = 'https://raw.githubusercontent.com/Jeffateth/XAllergen2.0/main'

    package_files = [
        '__init__.py',
        'baseline_notebook_utils.py',
        'baseline_sasa_experiment.py',
        'epitope_preprocessing.py',
        'mtl_epitope_notebook_utils.py',
        'rsa_preprocessing.py',
    ]
    data_files = [
        'deepalgpro_test_cleaned.csv',
        'positives_splitA.csv',
        'positives_splitB.csv',
        'negatives_splitA.csv',
        'negatives_splitB.csv',
    ]

    download_jobs = []
    download_jobs.extend((f'{RAW}/src/xallergen/{name}', PACKAGE_DIR / name) for name in package_files)
    download_jobs.extend((f'{RAW}/data/{name}', DATA_DIR / name) for name in data_files)
    download_jobs.append((f'{RAW}/models/baseline_frozen_esm2.pt', RUNTIME_MODELS_DIR / 'baseline_frozen_esm2.pt'))

    failed_urls = []
    for url, dst in download_jobs:
        try:
            urllib.request.urlretrieve(url, dst)
        except Exception as exc:
            failed_urls.append((url, str(exc)))

    if failed_urls:
        details = '\n'.join(f'  - {url} -> {err}' for url, err in failed_urls)
        raise RuntimeError('Failed to download required GitHub artifacts:\n' + details)

    fresh_model_path = snapshot_download(
        repo_id='facebook/esm2_t6_8M_UR50D',
        local_dir=HF_DIR,
        local_dir_use_symlinks=False,
        force_download=True,
        resume_download=False,
    )
    os.environ['XALLERGEN_HF_MODEL_DIR'] = str(fresh_model_path)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f'RUN_TARGET: {os.environ["XALLERGEN_RUN_TARGET"]}')
print(f'Runtime root: {RUNTIME_ROOT}')
print(f'Output results dir: {RESULTS_DIR}')

# IEDB Attention Regularization Sweep

Tests whether constraining pooling attention toward experimentally validated IEDB
epitope positions improves allergenicity classification and residue-level epitope
localisation — using the same training data as notebook 04 (MTL epitope supervision)
but replacing the separate epitope prediction head with a direct attention penalty.

**IEDB features ($e_i$)** are binary per-residue labels derived from
`positives_splitA.csv`: $e_i = 1$ if residue $i$ falls within at least one
annotated epitope interval, $e_i = 0$ otherwise.
Intervals are stored as semicolon-separated 1-based inclusive positions.

The regularization loss applies **only to allergen proteins** ($y = 1$). Non-allergens
receive no IEDB feature vector and are masked from the loss automatically.

λ = 0 is included as an in-sweep baseline (unregularized fine-tuning on the same
612-protein MTL dataset). Results are saved to `results/iedb_regularization/`
and compared against the baseline and MTL model (notebook 04) in Section C.

In [ ]:
import importlib
import sys
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'pyproject.toml').exists() or (candidate / 'src' / 'xallergen').exists():
            return candidate
    raise FileNotFoundError('Could not find repo root')


if 'DATA_DIR' not in globals():
    RUNTIME_ROOT   = _find_repo_root(Path.cwd())
    SRC_DIR        = RUNTIME_ROOT / 'src'
    DATA_DIR       = RUNTIME_ROOT / 'data'
    RUNTIME_MODELS_DIR = RUNTIME_ROOT / 'models'
    RESULTS_DIR    = RUNTIME_ROOT / 'results'
    if str(SRC_DIR) not in sys.path:
        sys.path.insert(0, str(SRC_DIR))

missing_modules = []
for module_name in ('numpy', 'pandas', 'torch', 'IPython', 'sklearn'):
    try:
        importlib.import_module(module_name)
    except ModuleNotFoundError:
        missing_modules.append(module_name)

if missing_modules:
    raise ModuleNotFoundError(
        'Missing kernel dependencies: ' + ', '.join(missing_modules) +
        '. Run `make setup` and select the `Python (xallergen2)` kernel.'
    )

import pandas as pd
import torch
from IPython.display import display
from sklearn.model_selection import train_test_split

from xallergen.baseline_notebook_utils import RANDOM_STATE, seed_everything
from xallergen.baseline_sasa_experiment import (
    RSARegularizationConfig,
    run_lambda_rsa_sweep,
)
from xallergen.epitope_preprocessing import build_iedb_residue_lookup

POSITIVES_A_CSV      = DATA_DIR / 'positives_splitA.csv'
POSITIVES_B_CSV      = DATA_DIR / 'positives_splitB.csv'
NEGATIVES_A_CSV      = DATA_DIR / 'negatives_splitA.csv'
NEGATIVES_B_CSV      = DATA_DIR / 'negatives_splitB.csv'
BASELINE_CHECKPOINT  = RUNTIME_ROOT / 'models' / 'baseline_frozen_esm2.pt'

IEDB_RESULTS_DIR = RESULTS_DIR / 'iedb_regularization'
IEDB_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
IEDB_SWEEP_CSV = IEDB_RESULTS_DIR / 'sweep_summary.csv'

for required_path in [
    POSITIVES_A_CSV, POSITIVES_B_CSV,
    NEGATIVES_A_CSV, NEGATIVES_B_CSV,
    BASELINE_CHECKPOINT,
]:
    if not required_path.exists():
        raise FileNotFoundError(f'Missing required file: {required_path}')

# λ=0 is included as the in-sweep unregularized SFT baseline.
# Range extends to 20.0 to check whether high regularization helps,
# consistent with the SS3-structured sweep (nb 13/15).
IEDB_CONFIG = RSARegularizationConfig(
    lambda_cls=1.0,
    lambda_rsa_values=(0.0, 0.1, 0.5, 1.0, 5.0, 10.0, 20.0),
    add_special_tokens=False,
    feature_key='iedb',
)

seed_everything(RANDOM_STATE)
DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'Feature: IEDB binary epitope annotations (1-based inclusive intervals)')
print(f'λ values: {IEDB_CONFIG.lambda_rsa_values}')
print(f'Output dir: {IEDB_RESULTS_DIR}')

---
## Section A — Data Loading and IEDB Feature Inspection

In [ ]:
pos_a = pd.read_csv(POSITIVES_A_CSV)
neg_a = pd.read_csv(NEGATIVES_A_CSV)
pos_b = pd.read_csv(POSITIVES_B_CSV)
neg_b = pd.read_csv(NEGATIVES_B_CSV)

def _build_frame(pos_df: pd.DataFrame, neg_df: pd.DataFrame) -> pd.DataFrame:
    """Combine positives and negatives into [sequence_id, sequence, label].

    Allergens use their UniProt accession as sequence_id (matching the IEDB lookup key).
    Non-allergens use their UniProt entry accession from the 'entry' column.
    """
    pos_out = pd.DataFrame({
        'sequence_id': pos_df['accession'].astype(str).str.strip(),
        'sequence':    pos_df['sequence'].astype(str).str.strip().str.upper(),
        'label':       1,
    })
    # Negatives: use 'entry' (UniProt accession) as sequence_id.
    # The value does not need to match IEDB data — negatives always get None.
    neg_ids = (
        neg_df['entry'].astype(str).str.strip()
        if 'entry' in neg_df.columns
        else pd.Series([f'neg_{i}' for i in range(len(neg_df))])
    )
    neg_out = pd.DataFrame({
        'sequence_id': neg_ids,
        'sequence':    neg_df['sequence'].astype(str).str.strip().str.upper(),
        'label':       0,
    })
    return pd.concat([pos_out, neg_out], ignore_index=True)


train_df = _build_frame(pos_a, neg_a)
test_df  = _build_frame(pos_b, neg_b)

print(f'Train: {len(train_df):,} proteins  ({(train_df["label"]==1).sum():,} allergens)')
print(f'Test:  {len(test_df):,}  proteins  ({(test_df["label"]==1).sum():,} allergens)')
print(f'Mean epitope coverage (trainA allergens): {pos_a["epitope_coverage"].mean():.3f}')
print(f'Mean epitope coverage (testB allergens):  {pos_b["epitope_coverage"].mean():.3f}')

In [ ]:
seed_everything(RANDOM_STATE)
train_split_df, val_split_df = train_test_split(
    train_df,
    test_size=0.1,
    random_state=RANDOM_STATE,
    stratify=train_df['label'],
)
train_split_df = train_split_df.reset_index(drop=True).copy()
val_split_df   = val_split_df.reset_index(drop=True).copy()

print(f'Train: {len(train_split_df):,} | Val: {len(val_split_df):,} | Test: {len(test_df):,}')

In [ ]:
# Build per-residue binary IEDB feature vectors.
# Allergens keyed by their UniProt accession; non-allergens receive None.
train_iedb_lookup = build_iedb_residue_lookup(POSITIVES_A_CSV, train_df)
test_iedb_lookup  = build_iedb_residue_lookup(POSITIVES_B_CSV, test_df)

# Coverage summary
train_n_with = sum(1 for v in train_iedb_lookup.values() if v is not None)
test_n_with  = sum(1 for v in test_iedb_lookup.values()  if v is not None)
print(f'Train: {train_n_with} proteins with IEDB vectors / {len(train_df)} total'
      f' ({train_n_with/(train_df["label"]==1).sum():.1%} of allergens)')
print(f'Test:  {test_n_with}  proteins with IEDB vectors / {len(test_df)} total'
      f' ({test_n_with/(test_df["label"]==1).sum():.1%} of allergens)')

# Quick sanity checks
sample_allergen_sid = train_df.loc[train_df['label'] == 1, 'sequence_id'].iloc[0]
vec = train_iedb_lookup[sample_allergen_sid]
print(f'\nSample allergen: {sample_allergen_sid}')
print(f'  Vector shape:    {vec.shape}')
print(f'  Epitope fraction:{float((vec > 0).float().mean()):.3f}')

sample_neg_sid = train_df.loc[train_df['label'] == 0, 'sequence_id'].iloc[0]
assert train_iedb_lookup[sample_neg_sid] is None, 'Non-allergen should have None'
print(f'Non-allergen {sample_neg_sid}: None ✓')

---
## Section B — IEDB Attention Regularization Sweep

λ values: **0.0 (baseline), 0.1, 0.5, 1.0, 5.0**.  
λ = 0 gives an unregularized fine-tuning run on the same 612-protein MTL dataset,
directly comparable to the regularized runs and to notebook 04 (MTL).
Each run starts from `baseline_frozen_esm2.pt`.

In [ ]:
sweep_df = run_lambda_rsa_sweep(
    config=IEDB_CONFIG,
    train_split_df=train_split_df,
    val_split_df=val_split_df,
    test_df=test_df,
    train_rsa_lookup=train_iedb_lookup,
    test_rsa_lookup=test_iedb_lookup,
    positives_splitb_csv=POSITIVES_B_CSV,
    output_dir=IEDB_RESULTS_DIR,
    device=DEVICE,
    baseline_checkpoint_path=BASELINE_CHECKPOINT,
).loc[:, [
    'lambda_rsa', 'epoch',
    'val_auroc', 'val_precision', 'val_recall', 'val_f1', 'val_mcc', 'val_accuracy',
    'test_auroc', 'test_precision', 'test_recall', 'test_f1', 'test_mcc', 'test_accuracy',
    'residue_auroc', 'residue_auprc', 'residue_precision_at_k',
]].copy()
sweep_df.to_csv(IEDB_SWEEP_CSV, index=False)
display(sweep_df)
print(f'Saved to: {IEDB_SWEEP_CSV}')
torch.cuda.empty_cache() if DEVICE == 'cuda' else None

In [ ]:
# Exclude λ=0 when picking best regularized run
reg_df = sweep_df.loc[sweep_df['lambda_rsa'].ne(0.0)]
baseline_row = sweep_df.loc[sweep_df['lambda_rsa'].eq(0.0)].iloc[0]
best = reg_df.loc[reg_df['val_mcc'].idxmax()]

print('=== IEDB attention regularization: in-sweep baseline (λ=0) ===')
print(f'  Val MCC      : {baseline_row["val_mcc"]:.4f}')
print(f'  Test MCC     : {baseline_row["test_mcc"]:.4f}')
print(f'  Residue AUROC: {baseline_row["residue_auroc"]:.4f}')
print()
print('=== Best regularized λ by val MCC ===')
print(f'  Best λ       : {best["lambda_rsa"]}')
print(f'  Val MCC      : {best["val_mcc"]:.4f}')
print(f'  Test MCC     : {best["test_mcc"]:.4f}')
print(f'  Test AUROC   : {best["test_auroc"]:.4f}')
print(f'  Residue AUROC: {best["residue_auroc"]:.4f}')

In [ ]:
# Evaluate every lambda checkpoint on the full deepalgpro test set (1,377 proteins).
# Non-allergens in this set have no IEDB annotations, so the lookup is all-None —
# no regularization is applied during inference, only classification metrics are computed.
import torch.nn as nn

from xallergen.baseline_notebook_utils import (
    DROPOUT, HIDDEN_DIM, HF_MODEL_NAME,
    build_tokenizer, load_baseline_checkpoint,
)
from xallergen.baseline_sasa_experiment import (
    build_dataloader, compute_metrics, format_lambda_suffix, predict,
)

DEEPALGPRO_TEST_CSV = DATA_DIR / 'deepalgpro_test_cleaned.csv'
if not DEEPALGPRO_TEST_CSV.exists():
    raise FileNotFoundError(
        f'Missing: {DEEPALGPRO_TEST_CSV}\n'
        'On Colab this is downloaded automatically; locally it should already be present.'
    )

deepalgpro_df = pd.read_csv(DEEPALGPRO_TEST_CSV).copy()
deepalgpro_df['sequence_id'] = deepalgpro_df['sequence_id'].astype(str).str.strip()
deepalgpro_df['sequence']    = deepalgpro_df['sequence'].astype(str).str.strip().str.upper()
deepalgpro_df['label']       = deepalgpro_df['label'].astype(int)

tokenizer = build_tokenizer(HF_MODEL_NAME)
# All proteins in the deepalgpro test set get None (no IEDB annotation available).
deepalgpro_lookup = {sid: None for sid in deepalgpro_df['sequence_id']}
deepalgpro_loader = build_dataloader(
    deepalgpro_df, deepalgpro_lookup, tokenizer,
    batch_size=IEDB_CONFIG.batch_size,
    shuffle=False,
    add_special_tokens=IEDB_CONFIG.add_special_tokens,
)

deepalgpro_rows = []
for lambda_val in IEDB_CONFIG.lambda_rsa_values:
    suffix   = format_lambda_suffix(lambda_val)
    ckpt     = IEDB_RESULTS_DIR / f'lambda_{suffix}' / 'baseline_frozen_esm2.pt'
    if not ckpt.exists():
        print(f'Checkpoint not found: {ckpt}')
        continue
    model, _ = load_baseline_checkpoint(
        ckpt, DEVICE,
        model_name=HF_MODEL_NAME,
        hidden_dim=IEDB_CONFIG.hidden_dim,
        dropout=IEDB_CONFIG.dropout,
    )
    _, pred_df = predict(model, deepalgpro_loader, DEVICE, threshold=IEDB_CONFIG.threshold)
    m = compute_metrics(pred_df, threshold=IEDB_CONFIG.threshold)
    deepalgpro_rows.append({'lambda_rsa': lambda_val, **m})
    del model
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

deepalgpro_sweep_df = pd.DataFrame(deepalgpro_rows)
print('=== DeepAlgPro test set (1,377 proteins) — all lambda checkpoints ===')
display(deepalgpro_sweep_df[['lambda_rsa', 'mcc', 'auroc', 'auprc', 'f1', 'accuracy']].round(4))

---
## Section C — Comparison Tables

Two test sets are reported:
- **SplitB test set** (136 proteins = positives_splitB + negatives_splitB): the held-out MTL test set, directly comparable to notebook 04 (MTL).
- **DeepAlgPro test set** (1,377 proteins): the full protein-level classification benchmark, directly comparable to all other sweeps (notebooks 12–16).

All models start from `baseline_frozen_esm2.pt`.

In [ ]:
import json

MTL_METRICS_JSON = RESULTS_DIR / 'classification' / 'mtl_baseline_metrics.json'


# ── Table 1: SplitB test set (MTL comparison) ───────────────────────────────

def _row_from_df(df: pd.DataFrame, label: str, lambda_val: float | None = None) -> dict:
    row = df.loc[df['lambda_rsa'].eq(lambda_val)].iloc[0] if lambda_val is not None \
          else df.loc[df['val_mcc'].idxmax()]
    return {
        'Model':          label,
        'Best λ':         float(row['lambda_rsa']),
        'Val MCC':        round(float(row['val_mcc']),              4),
        'SplitB MCC':     round(float(row['test_mcc']),             4),
        'SplitB AUROC':   round(float(row['test_auroc']),           4),
        'Residue AUROC':  round(float(row['residue_auroc']),        4),
        'Residue AUPRC':  round(float(row['residue_auprc']),        4),
        'Residue Prec@k': round(float(row['residue_precision_at_k']), 4),
    }


splitb_rows = [
    _row_from_df(sweep_df, 'λ=0 (unregularized, MTL data)', lambda_val=0.0),
    _row_from_df(reg_df,   'IEDB attention reg (best λ)'),
]

if MTL_METRICS_JSON.exists():
    with MTL_METRICS_JSON.open() as fh:
        mtl = json.load(fh)
    splitb_key = next((k for k in mtl if 'split' in k.lower() and 'b' in k.lower()), None)
    splitb     = mtl.get(splitb_key, {}) if splitb_key else {}
    res_src    = mtl.get('residue_metrics_splitb', mtl.get('test_residue_metrics', {}))
    splitb_rows.append({
        'Model':          'MTL (nb 04)',
        'Best λ':         mtl.get('lambda_epi', 0.5),
        'Val MCC':        round(float(mtl.get('val_metrics', {}).get('mcc', float('nan'))), 4),
        'SplitB MCC':     round(float(splitb.get('mcc',   float('nan'))), 4),
        'SplitB AUROC':   round(float(splitb.get('auroc', float('nan'))), 4),
        'Residue AUROC':  round(float(res_src.get('auroc',           float('nan'))), 4),
        'Residue AUPRC':  round(float(res_src.get('auprc',           float('nan'))), 4),
        'Residue Prec@k': round(float(res_src.get('precision_at_k',  float('nan'))), 4),
    })
else:
    print(f'MTL metrics not found at {MTL_METRICS_JSON} — run notebook 04 first.')

splitb_df = pd.DataFrame(splitb_rows).set_index('Model')
print('=== Table 1: SplitB test set (positives_splitB + negatives_splitB, 136 proteins) ===')
display(splitb_df)


# ── Table 2: DeepAlgPro test set (full benchmark) ───────────────────────────

def _row_deepalgpro(lambda_val: float, label: str) -> dict:
    row = deepalgpro_sweep_df.loc[deepalgpro_sweep_df['lambda_rsa'].eq(lambda_val)].iloc[0]
    return {
        'Model':       label,
        'Best λ':      lambda_val,
        'DA MCC':      round(float(row['mcc']),     4),
        'DA AUROC':    round(float(row['auroc']),   4),
        'DA AUPRC':    round(float(row['auprc']),   4),
        'DA F1':       round(float(row['f1']),      4),
        'DA Accuracy': round(float(row['accuracy']), 4),
    }


best_lambda = float(reg_df.loc[reg_df['val_mcc'].idxmax(), 'lambda_rsa'])
deepalgpro_rows_cmp = [
    _row_deepalgpro(0.0,          'λ=0 (unregularized, MTL data)'),
    _row_deepalgpro(best_lambda,  f'IEDB attention reg (λ={best_lambda:g})'),
]

# Add nb 04 deepalgpro metrics if stored in the JSON
if MTL_METRICS_JSON.exists():
    da_src = mtl.get('test_metrics', {})
    if da_src.get('dataset_name', '') == 'deepalgpro_test_cleaned':
        deepalgpro_rows_cmp.append({
            'Model':       'MTL (nb 04)',
            'Best λ':      mtl.get('lambda_epi', 0.5),
            'DA MCC':      round(float(da_src.get('mcc',      float('nan'))), 4),
            'DA AUROC':    round(float(da_src.get('auroc',    float('nan'))), 4),
            'DA AUPRC':    round(float(da_src.get('auprc',    float('nan'))), 4),
            'DA F1':       round(float(da_src.get('f1',       float('nan'))), 4),
            'DA Accuracy': round(float(da_src.get('accuracy', float('nan'))), 4),
        })

deepalgpro_cmp_df = pd.DataFrame(deepalgpro_rows_cmp).set_index('Model')
print('\n=== Table 2: DeepAlgPro test set (deepalgpro_test_cleaned.csv, 1,377 proteins) ===')
display(deepalgpro_cmp_df)

In [ ]:
# Shut down the Colab runtime to free GPU after training completes.
if os.environ.get('XALLERGEN_RUN_TARGET') == 'colab':
    import os as _os
    _os.kill(_os.getpid(), 9)